# Resuming: MQTT + OD2000 Integration (`feat-mqtt-od2000-integration`)

**Written:** 2026-07-18
**Updated:** 2026-07-28 — OD2000 installed and confirmed live via HTTP polling. **Later the same day: scanning architecture unified** — `scan_runner.py` retired, its logic merged into `gantry_agent.py` (now the sole, permanent owner of the BLC serial port, running scans on a background thread with a live STOP path). See `docs/MACRON_GANTRY.md` and `docs/RANGEFINDER_PROFILING.md`.
**Branch:** `feat-mqtt-od2000-integration` (off `develop`)
**Status:** All of Steps 1–7 confirmed/resolved. `TopographicProfiler` rewritten as a thin coordinator over `gantry.connection.start_scan()`/`wait_for_scan_result()`/`stop_scan()` — no more separate deployed script, no more disconnect/reconnect dance. **308 tests pass.** Step 8 (real scan on hardware) has not yet been run for real — the code path is new, offline-tested only.

---

## What was built

| File | Purpose |
|------|---------|
| `src/laguna/pi/gauge_publisher.py` | Standalone Pi script: reads Massa gauge over serial, publishes to `laguna/gauge/water_level_mm` |
| `src/laguna/robot/macron/gantry_agent.py` | Pi-resident agent: interactive axis commands AND full topographic scans (background thread, OD2000 HTTP polling, laser on/off, live STOP) — sole owner of the serial port |
| `src/laguna/robot/macron/pi_bridge.py` | `PiGantryConnection` — background reader thread dispatches responses; `start_scan()`/`stop_scan()`/`wait_for_scan_result()` |
| `src/laguna/mqtt/__init__.py` | `MqttSubscriber` — paho-mqtt wrapper (used for gauge publisher + optional low-rate monitoring, not scanning) |
| `src/laguna/rangefinder/__init__.py` | `RangefinderSubsystem` + `decode_od2000_pdin()` — decoder confirmed correct against physical reference 2026-07-28 |
| `src/laguna/robot/macron/profiler.py` | `TopographicProfiler` — thin coordinator over the gantry's own agent connection; no more separate SSH session or SFTP script deploy |
| `docs/MQTT_AL1342_SETUP.md` | AL1342 hardware bring-up guide with the HTTP polling strategy and all confirmed facts |
| `docs/MACRON_GANTRY.md` | Updated with the unified scanning architecture and `pi_agent`-transport requirement |
| `docs/subsystems/rangefinder.md` | Narrative article — updated for the new scan API |
| `tests/test_gantry_agent.py` | 39 tests: serial locking, PDIN decode, laser control, HTTP polling, dead-reckoning, `_run_scan` worker (STOP, error, laser-off-on-error) |
| `tests/test_macron_pi_bridge.py` | 56 tests: reader-thread dispatch, scan control API, safe-mode gate |
| `tests/test_rangefinder.py` | 28 tests: PDIN decoding, JSON path, RangefinderSubsystem |
| `tests/test_mqtt_subscriber.py` | 30 tests: MqttSubscriber lifecycle and message buffering |
| `tests/test_profiler.py` | 14 tests: TopographicProfiler orchestration against a fake gantry connection |

**scan_runner.py and test_scan_runner_logic.py were deleted** — their logic lives in gantry_agent.py/test_gantry_agent.py now.

---

## Network / hostname recap — confirmed 2026-07-27

In [1]:
import subprocess

result = subprocess.run(
    ["git", "branch", "--show-current"],
    capture_output=True, text=True, cwd="/home/eric/mysoftware/laguna"
)
branch = result.stdout.strip()
print(f"Current branch: {branch}")
assert branch == "feat-mqtt-od2000-integration", (
    f"Expected feat-mqtt-od2000-integration, got {branch!r}. "
    "Run: git checkout feat-mqtt-od2000-integration"
)

Current branch: feat-mqtt-od2000-integration


In [3]:
# Confirm tests still pass after any changes made since the branch was created
result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-q", "--no-header", "--tb=short"],
    capture_output=True, text=True, cwd="/home/eric/mysoftware/laguna"
)
print(result.stdout[-3000:])  # last 3000 chars
if result.returncode != 0:
    print(result.stderr[-1000:])


ERROR: usage: python -m pytest [options] [file_or_dir] [file_or_dir] [...]
python -m pytest: error: unrecognized arguments: --cov=src/laguna --cov-report=html
  inifile: /home/eric/mysoftware/laguna/pyproject.toml
  rootdir: /home/eric/mysoftware/laguna




---

## Step 1 — Install Mosquitto on the Pi — confirmed working ✓

Do this once over SSH before anything else.

```bash
ssh oak@red.lab
sudo apt update && sudo apt install -y mosquitto mosquitto-clients
sudo systemctl enable mosquitto && sudo systemctl start mosquitto
```

**Pitfall hit on hardware:** the default `/etc/mosquitto/mosquitto.conf` on this Pi
already sets `log_dest file ...` at line 11. Adding the same `log_dest` line again in
a `conf.d/` override causes Mosquitto to refuse to start (`Duplicate "log_dest" value`,
exit status 3). Check first:

```bash
grep -n "log_dest" /etc/mosquitto/mosquitto.conf
```

If already set, create `/etc/mosquitto/conf.d/laguna.conf` **without** a `log_dest` line:
```
listener 1883
allow_anonymous true
```

Then `sudo systemctl restart mosquitto`.

**Verify:** The cell below SSHes to the Pi and checks Mosquitto status.

In [4]:
import paramiko

PI_HOST = "red.lab"
PI_USER = "oak"
PI_KEY  = "/home/eric/.ssh/id_ed25519"

def ssh_run(cmd, host=PI_HOST, user=PI_USER, key=PI_KEY):
    client = paramiko.SSHClient()
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    client.connect(host, username=user, key_filename=key)
    _, stdout, stderr = client.exec_command(cmd)
    out = stdout.read().decode()
    err = stderr.read().decode()
    client.close()
    return out, err

out, err = ssh_run("systemctl is-active mosquitto")
print("mosquitto:", out.strip())
assert out.strip() == "active", "Mosquitto is not running — follow Step 1 setup above"

mosquitto: active


---

## Step 2 — Network addressing — confirmed working ✓

The AL1342 currently has a **static IP** (`192.168.1.251`) set directly on the device —
it is not on dnsmasq DHCP right now, and that's fine, no DHCP reservation work needed.

Add an `/etc/hosts` entry on the laguna PC so `al1342.lab` resolves locally
(this only helps *our* requests to the device — see the note in the intro
about why the AL1342 itself needs raw IPs, not hostnames):

```
192.168.1.251    al1342.lab al1342
```

No dnsmasq restart needed — `/etc/hosts` takes precedence and is picked up immediately.

**Verify:**

In [5]:
import socket

AL1342_HOST = "al1342.lab"

try:
    ip = socket.gethostbyname(AL1342_HOST)
    print(f"al1342.lab resolves to {ip} ✓")
except socket.gaierror as e:
    print(f"Cannot resolve al1342.lab: {e}")
    print("→ Follow Step 2 to add the DHCP reservation and A-record")

al1342.lab resolves to 192.168.1.251 ✓


In [7]:
# Check the IoT-Core Visualizer is reachable
import urllib.request

try:
    with urllib.request.urlopen(f"http://{AL1342_HOST}", timeout=5) as resp:
        print(f"IoT-Core Visualizer HTTP {resp.status} ✓")
except Exception as e:
    print(f"Cannot reach AL1342 web UI: {e}")
    print("→ Check physical Ethernet connection and IP assignment")

IoT-Core Visualizer HTTP 200 ✓


---

## Step 3 — Confirm OD2000 on AL1342

Open `http://al1342.lab/web/subscribe` in a browser, go to **Parameter → Iolinkmaster**, and find the port the OD2000 is plugged into. Note the port number — you'll use it everywhere as `pdin_port`.

Expected fields:
- `vendorid` = 85  
- `productname` contains "OD2000"

**Set `PDIN_PORT` here and carry it through the rest of the notebook:**

In [ ]:
PDIN_PORT = 2  # Confirmed 2026-07-28: OD2000-7002T15 found on port 2 (vendorid=26, not 85 as guessed)
print(f"OD2000 is on IO-Link port {PDIN_PORT}")

---

## Step 4 — AL1342 HTTP control interface — confirmed working ✓

**Key correction:** there is no MQTT-based "command channel" required to control
the AL1342. All configuration (`setdata`) and subscription requests (`subscribe`)
are plain HTTP POST requests to the device root — `http://192.168.1.251/`.

The AL1342 has an *optional* extra feature (`mqttCmdChannel`) that lets it also
receive the same commands over MQTT. We enable it below since it seemed required
at first, but **HTTP POST works reliably for everything and is the default we use
throughout this notebook.** The MQTT command channel itself (publish to `cmdTopic`,
read `defaultReplyTopic`) was never actually exercised — treat it as unverified,
redundant infrastructure.

Two bugs from the original plan are fixed in the cell below:
1. There is **no `MQTTSetup` path segment** — the real path is
   `/connections/mqttConnection/mqttCmdChannel/...`, not
   `/connections/mqttConnection/MQTTSetup/mqttCmdChannel/...` (this caused the 404 below)
2. `brokerPort` is typed `number`/`integer` — sending it as a string (`"1883"`)
   returns code 400; it must be `{"newvalue": 1883}` (this caused the 400 below)

See `docs/MQTT_AL1342_SETUP.md` Step 4 for the full explanation.

**First attempt (kept for the record — shows the failure modes above):**

In [8]:
import json, urllib.request

# BROKER_ADDR = "red.lab"   # ← substitute Pi IP if hostname doesn't work in AL1342
BROKER_ADDR = "192.168.1.58"

AL1342_URL  = f"http://{AL1342_HOST}/iolinkmaster"

def al1342_post(payload: dict) -> dict:
    data = json.dumps(payload).encode()
    req = urllib.request.Request(AL1342_URL, data=data,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=5) as resp:
        return json.loads(resp.read())

bootstrap_cmds = [
    {"code": "request", "cid": 1,
     "adr": "/connections/mqttConnection/MQTTSetup/mqttCmdChannel/status/start"},
    {"code": "request", "cid": 2,
     "adr": "/connections/mqttConnection/mqttCmdChannel/mqttCmdChannelSetup/brokerIP/setdata",
     "data": {"newvalue": BROKER_ADDR}},
    {"code": "request", "cid": 3,
     "adr": "/connections/mqttConnection/mqttCmdChannel/mqttCmdChannelSetup/brokerPort/setdata",
     "data": {"newvalue": "1883"}},
    {"code": "request", "cid": 4,
     "adr": "/connections/mqttConnection/mqttCmdChannel/mqttCmdChannelSetup/cmdTopic/setdata",
     "data": {"newvalue": "laguna/al1342/cmd"}},
    {"code": "request", "cid": 5,
     "adr": "/connections/mqttConnection/mqttCmdChannel/mqttCmdChannelSetup/defaultReplyTopic/setdata",
     "data": {"newvalue": "laguna/al1342/reply"}},
]

for cmd in bootstrap_cmds:
    try:
        resp = al1342_post(cmd)
        code = resp.get("code", "?")
        print(f"cid={cmd['cid']} → {code}")
    except Exception as e:
        print(f"cid={cmd['cid']} FAILED: {e}")

cid=1 → 404
cid=2 → 200
cid=3 → 400
cid=4 → 200
cid=5 → 200


In [ ]:
AL1342_IP   = "192.168.1.251"
AL1342_URL  = f"http://{AL1342_IP}/"   # base root — NOT /iolinkmaster, that was the earlier bug
BROKER_IP   = "192.168.1.58"           # Pi's actual IP — AL1342 must use IP, not hostname
PDIN_PORT   = 1                         # update once OD2000 is physically installed (Step 3)
OD2000_TOPIC = "laguna/od2000"

import json, urllib.request

def al1342_post(payload: dict) -> dict:
    data = json.dumps(payload).encode()
    req = urllib.request.Request(AL1342_URL, data=data,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=5) as resp:
        return json.loads(resp.read())

# Discover the device tree — always do this rather than trusting manual path names;
# this firmware's tree differs from at least one documented example.
resp = al1342_post({"code": "request", "cid": -1, "adr": "gettree"})
with open("/tmp/al1342_tree.json", "w") as f:
    json.dump(resp, f)
print(f"Tree fetched: {len(json.dumps(resp))} bytes, saved to /tmp/al1342_tree.json")
print(json.dumps(resp, indent=2)[:1500])

In [ ]:
# Corrected bootstrap: fixed paths (no MQTTSetup segment), brokerPort as integer
bootstrap_cmds = [
    {"code": "request", "cid": 2,
     "adr": "/connections/mqttConnection/mqttCmdChannel/mqttCmdChannelSetup/brokerIP/setdata",
     "data": {"newvalue": BROKER_IP}},
    {"code": "request", "cid": 3,
     "adr": "/connections/mqttConnection/mqttCmdChannel/mqttCmdChannelSetup/brokerPort/setdata",
     "data": {"newvalue": 1883}},   # integer, not string — string caused code 400 earlier
    {"code": "request", "cid": 4,
     "adr": "/connections/mqttConnection/mqttCmdChannel/mqttCmdChannelSetup/cmdTopic/setdata",
     "data": {"newvalue": "laguna/al1342/cmd"}},
    {"code": "request", "cid": 5,
     "adr": "/connections/mqttConnection/mqttCmdChannel/mqttCmdChannelSetup/defaultReplyTopic/setdata",
     "data": {"newvalue": "laguna/al1342/reply"}},
    {"code": "request", "cid": 1,
     "adr": "/connections/mqttConnection/mqttCmdChannel/status/start",  # no MQTTSetup segment
     "data": {}},
]

for cmd in bootstrap_cmds:
    try:
        resp = al1342_post(cmd)
        code = resp.get("code", "?")
        print(f"cid={cmd['cid']} → {code}")
    except Exception as e:
        print(f"cid={cmd['cid']} FAILED: {e}")

# Confirm the command channel actually connected
status_resp = al1342_post({"code": "request", "cid": 30,
                            "adr": "/connections/mqttConnection/mqttCmdChannel/status/getdata"})
print(f"\nCommand channel status: {status_resp}")
assert status_resp.get("data", {}).get("value") == "running", (
    "Command channel did not report 'running' — check the Mosquitto log on the Pi "
    "for a new connection from the AL1342's IP."
)
print("\n✓ Confirmed: check the Pi's Mosquitto log for a client connection from "
      f"{AL1342_IP} — the AL1342 connects using its own MAC address as client ID.")

In [ ]:
# Utility: find every path in the device tree that supports datachanged subscription.
# Reused in Step 6 to check whether the OD2000's pdin has its own datachanged event.
def find_datachanged_paths(tree_data):
    paths = []
    def walk(node, path):
        ident = node.get("identifier", "")
        newpath = path + "/" + ident if ident else path
        subs = node.get("subs", [])
        names = [s.get("identifier") for s in subs]
        if "datachanged" in names:
            paths.append(newpath)
        for s in subs:
            walk(s, newpath)
    walk(tree_data, "")
    return paths

with open("/tmp/al1342_tree.json") as f:
    tree = json.load(f)

dc_paths = find_datachanged_paths(tree["data"])
print("Paths with datachanged subelement:")
for p in dc_paths:
    print(" ", p)

print(
    "\nNote: /processdatamaster/temperature and pdin-style process values are "
    "NOT in this list on this firmware — only timer[n]/counter (periodic, 500ms "
    "floor), portevent, and iolinkevent (discrete state-change events) support "
    "datachanged. See docs/MQTT_AL1342_SETUP.md Step 5 for what this means."
)

---

## Step 5 — Pipeline wiring test (temperature, no OD2000 needed) — confirmed working ✓

Before the OD2000 is even installed, this proves the full chain works:
**AL1342 → Mosquitto on the Pi → our subscriber.**

Since `temperature` has no `datachanged` of its own (confirmed above), we use
the one push mechanism that's always available: subscribe `timer[1]` to tick
and carry `temperature` along in `datatosend`.

**Confirmed on hardware:** 24 messages in 12.0 s ≈ 2.00 Hz — matches the
documented 500 ms floor almost exactly. This is a *timer* constraint, not a
general MQTT-push constraint — see the note in the previous cell's output
about `portevent`/`iolinkevent` having no such floor (they just don't carry
continuous data either).

Note: `paho-mqtt` is not installed in this notebook's kernel — we use
`mosquitto_sub` via subprocess instead, which needs no Python dependency.

In [ ]:
import subprocess, time

TEMP_TOPIC = "laguna/al1342/temp"

# Subscribe timer[1] to push temperature every tick
subscribe_resp = al1342_post({
    "code": "request", "cid": 40,
    "adr": "/timer[1]/counter/datachanged/subscribe",
    "data": {
        "callback": f"mqtt://{BROKER_IP}:1883/{TEMP_TOPIC}",
        "datatosend": ["/processdatamaster/temperature"],
    },
})
print("Subscribe response:", subscribe_resp)

interval_resp = al1342_post({
    "code": "request", "cid": 41,
    "adr": "/timer[1]/interval/setdata",
    "data": {"newvalue": 500},   # documented + measured floor
})
print("Interval response:", interval_resp)

# Collect for 12s via mosquitto_sub (no paho needed)
print("\nCollecting for 12 s …")
result = subprocess.run(
    ["timeout", "12", "mosquitto_sub", "-h", BROKER_IP, "-t", TEMP_TOPIC, "-v"],
    capture_output=True, text=True,
)
lines = [l for l in result.stdout.splitlines() if l.strip()]
print(f"\nMessages received: {len(lines)}")
if len(lines) > 1:
    rate = len(lines) / 12.0
    print(f"Achieved rate: {rate:.2f} Hz  (expect ~2.0 Hz at 500ms interval)")
if lines:
    print(f"\nLatest message:\n{lines[-1]}")

---

## Step 6 — OD2000-specific verification — confirmed working ✓ (2026-07-28)

Live results, in order:

1. **Port confirmed**: walked `iolinkmaster/port[n]/iolinkdevice` for n=1–8,
   read `vendorid`/`productname` on each. Found the OD2000
   (`productname=OD2000-7002T15`) on **port 2**. `vendorid=26`, not 85 as
   the manual-derived guess assumed — match on `productname`, not `vendorid`.

2. **`pdin` confirmed to have no direct `datachanged`** — same pattern as
   `processdatamaster/temperature` from Step 5. Only `port[2]/portevent` and
   `port[2]/iolinkdevice/iolinkevent` support it, and neither carries
   continuous distance data. So MQTT push for OD2000 data is capped at the
   `timer[1]` 500 ms / 2 Hz floor, same as temperature — confirmed by
   design, not just by inference from the temperature test.

3. **Decoder validated against a physical reference.** With the OD2000
   reading a physically measured 808.4 mm ± 0.1 mm, a single `pdin/getdata`
   read returned `302D56F7F700`. Big-endian nm decode → **808.2778 mm** —
   matches within noise. Little-endian gives a negative (impossible) value,
   ruling that out unambiguously. One anomaly: byte 4 ("scale") = 247, not 0
   as assumed "normal" — unused in the decode, doesn't affect correctness,
   but unexplained.

4. **The 2 Hz ceiling was solved by switching to HTTP polling, not MQTT at
   all** — see 6.4 below. `scan_runner.py` and `TopographicProfiler` were
   rewritten to use this as the only OD2000 data path.

In [ ]:
# 6.1 — Confirmed 2026-07-28: OD2000 is on port 2 (productname=OD2000-7002T15, vendorid=26)
resp = al1342_post({"code": "request", "cid": -1, "adr": "gettree"})
with open("/tmp/al1342_tree.json", "w") as f:
    json.dump(resp, f)
tree = resp

def find_od2000_port(tree_data):
    def walk(node, path):
        ident = node.get("identifier", "")
        newpath = path + "/" + ident if ident else path
        subs = node.get("subs", [])
        # Look for iolinkdevice nodes and check their vendorid/productname via getdata separately
        if newpath.endswith("iolinkdevice"):
            yield newpath
        for s in subs:
            yield from walk(s, newpath)
    return list(walk(tree_data, ""))

device_paths = find_od2000_port(tree["data"])
print("iolinkdevice paths found:")
for p in device_paths:
    port_resp = al1342_post({"code": "request", "cid": -1, "adr": f"{p}/vendorid/getdata"})
    name_resp = al1342_post({"code": "request", "cid": -1, "adr": f"{p}/productname/getdata"})
    print(f"  {p}: vendorid={port_resp.get('data', {}).get('value')} "
          f"productname={name_resp.get('data', {}).get('value')}")

PDIN_PORT = 2  # Confirmed 2026-07-28
print(f"\nOD2000 confirmed on IO-Link port {PDIN_PORT}")

In [ ]:
# 6.2 — Check whether pdin has its own datachanged, and subscribe accordingly
pdin_path = f"/iolinkmaster/port[{PDIN_PORT}]/iolinkdevice/pdin"

dc_paths_fresh = find_datachanged_paths(tree["data"])
pdin_has_own_event = any(p.endswith("/pdin") for p in dc_paths_fresh)
print(f"pdin datachanged paths in tree: {[p for p in dc_paths_fresh if 'pdin' in p] or 'none'}")
print(f"pdin has its own datachanged event: {pdin_has_own_event}")

if pdin_has_own_event:
    print("\n→ Subscribing directly to pdin/datachanged (bypasses the timer, may be much faster)")
    subscribe_resp = al1342_post({
        "code": "request", "cid": 50,
        "adr": f"{pdin_path}/datachanged/subscribe",
        "data": {
            "callback": f"mqtt://{BROKER_IP}:1883/{OD2000_TOPIC}",
            "datatosend": [pdin_path],
        },
    })
    print("Subscribe response:", subscribe_resp)
    if subscribe_resp.get("code") != 200:
        print("→ Direct subscribe failed despite appearing in the tree — falling back to timer method below")
        pdin_has_own_event = False

if not pdin_has_own_event:
    print("\n→ No direct datachanged on pdin — using timer[1] fallback (500ms / 2Hz floor, confirmed reliable)")
    subscribe_resp = al1342_post({
        "code": "request", "cid": 10,
        "adr": "/timer[1]/counter/datachanged/subscribe",
        "data": {
            "callback": f"mqtt://{BROKER_IP}:1883/{OD2000_TOPIC}",
            "datatosend": [pdin_path],
        },
    })
    print("Subscribe response:", subscribe_resp)
    interval_resp = al1342_post({
        "code": "request", "cid": 11,
        "adr": "/timer[1]/interval/setdata",
        "data": {"newvalue": 500},
    })
    print("Interval response:", interval_resp)

In [ ]:
# 6.3 — Collect and decode. Compare against a known physical target distance.
import subprocess, sys
sys.path.insert(0, "/home/eric/mysoftware/laguna/src")
from laguna.rangefinder import decode_od2000_pdin

COLLECT_SECONDS = 12 if not pdin_has_own_event else 5  # timer path needs longer to get enough samples

print(f"Collecting for {COLLECT_SECONDS} s …")
result = subprocess.run(
    ["timeout", str(COLLECT_SECONDS), "mosquitto_sub", "-h", BROKER_IP, "-t", OD2000_TOPIC, "-v"],
    capture_output=True, text=True,
)
lines = [l for l in result.stdout.splitlines() if l.strip()]
print(f"Messages received: {len(lines)}")
if len(lines) > 1:
    rate = len(lines) / COLLECT_SECONDS
    print(f"Achieved rate: {rate:.2f} Hz")

decoded = []
errors = []
for line in lines:
    # line format: "<topic> <json>"
    _, _, json_str = line.partition(" ")
    try:
        payload = json.loads(json_str)
        hex_str = payload["data"]["payload"][pdin_path]["data"]
        decoded.append({**decode_od2000_pdin(hex_str), "hex": hex_str})
    except Exception as e:
        errors.append({"error": str(e), "raw": line[:200]})

print(f"\nDecoded OK: {len(decoded)}   Decode errors: {len(errors)}")
if decoded:
    dists = [d["distance_mm"] for d in decoded]
    print(f"Distance range: {min(dists):.1f} – {max(dists):.1f} mm")
    print(f"Latest sample: {decoded[-1]}")
    print(
        "\nCompare distance_mm against a known physical target distance:\n"
        "  - Matches            → decoder assumption (big-endian nm) confirmed\n"
        "  - Off by 1000x       → AL1342 publishes µm, not nm — divide by 1000 instead\n"
        "  - Wildly wrong/negative → try little-endian:\n"
        "      int.from_bytes(bytes.fromhex(hex_str)[0:4], 'little', signed=True) / 1e6"
    )
elif errors:
    print(f"\nFirst error: {errors[0]}")
    print("Check: PDIN_PORT correct? Subscribe actually succeeded above? pdin_path matches AL1342's exact path?")

### 6.4 — HTTP polling — confirmed working ✓ (2026-07-28)

The `timer[1]` 2 Hz ceiling (confirmed above) was solved by switching to
plain HTTP polling of `pdin/getdata` — no subscribe, no MQTT at all.

**First attempt failed**: looping `al1342_post()` (which uses `urllib`,
opening a fresh TCP connection every call) timed out after a couple of
seconds — the tight-loop connection churn choked the AL1342's embedded HTTP
server. A single request immediately afterward still worked fine, confirming
the device itself was healthy; the *pattern* was the problem.

**Fix: reuse one persistent `http.client.HTTPConnection`.** Result:
**1904 samples in 5.00 s = 380.7 Hz, zero errors** — ~190x faster than the
MQTT timer path. Distance range 807.98–808.59 mm, mean 808.28 mm, matching
the known 808.4 mm reference.

This is now how `scan_runner.py` collects OD2000 data during a scan (see
`_poll_pdin_loop()` — runs on a background thread with reconnect-on-error).
`TopographicProfiler`'s `od2000_topic` constructor arg was replaced with
`al1342_host` (a raw IP). Modbus TCP and the hardware capture-latch
(Approach B) were considered but not needed — see
`docs/MQTT_AL1342_SETUP.md` for the full comparison.

In [ ]:
# 6.4 — Poll pdin/getdata over a PERSISTENT connection (not al1342_post/urllib —
# that opens a new TCP connection per call and chokes the AL1342 in a tight loop)
import http.client, time
from laguna.rangefinder import decode_od2000_pdin

POLL_SECONDS = 5
pdin_path_getdata = f"/iolinkmaster/port[{PDIN_PORT}]/iolinkdevice/pdin/getdata"

conn = http.client.HTTPConnection(AL1342_IP, 80, timeout=5)
payload = json.dumps({"code": "request", "cid": -1, "adr": pdin_path_getdata})
headers = {"Content-Type": "application/json"}

samples = []
errors = 0
t_start = time.time()
while time.time() - t_start < POLL_SECONDS:
    try:
        conn.request("POST", "/", body=payload, headers=headers)
        resp = conn.getresponse()
        data = json.loads(resp.read())
        hex_str = data.get("data", {}).get("value")
        if hex_str:
            samples.append({"wall_time": time.time(), "hex": hex_str, **decode_od2000_pdin(hex_str)})
    except Exception as e:
        errors += 1
        conn.close()
        conn = http.client.HTTPConnection(AL1342_IP, 80, timeout=5)
conn.close()

elapsed = time.time() - t_start
rate = len(samples) / elapsed if elapsed > 0 else 0
print(f"Samples: {len(samples)} in {elapsed:.2f}s → {rate:.1f} Hz, errors: {errors}")
if samples:
    dists = [s["distance_mm"] for s in samples]
    print(f"Distance range: {min(dists):.4f} – {max(dists):.4f} mm")
    print(f"Mean: {sum(dists)/len(dists):.4f} mm")
    print(f"Latest: {samples[-1]}")

---

## Step 7 — Check serial_bridge.py port behavior — resolved, architecture changed

**Update (later 2026-07-28):** rather than working around this conflict per-scan
(the original disconnect/reconnect dance), the architecture was changed so
`gantry_agent.py` is the *permanent* sole owner of the serial port for the whole
session, and scanning now runs *inside* `gantry_agent.py` itself (a background
thread, sharing the same locked serial connection as interactive commands) —
see `docs/MACRON_GANTRY.md` and `docs/RANGEFINDER_PROFILING.md`. `scan_runner.py`
was retired; `TopographicProfiler` no longer touches the serial port question at
all.

This means the real requirement is simpler than "handle the conflict": **use
the `pi_agent` transport, not `socket_bridge`,** and make sure `serial_bridge.py`
is not running (confirmed from its source: it opens the serial device once and
never releases it — the two must never run concurrently, full stop, no
workaround needed or possible).

```yaml
# config/example_config.yaml — gantry: section
transport: pi_agent   # NOT socket_bridge (that's serial_bridge.py, incompatible with scanning)
```

The cell below still checks whether `serial_bridge.py` is running, since that's
the thing to stop before using `pi_agent`/scanning.

In [ ]:
# Check whether serial_bridge.py currently holds the serial port open
out, err = ssh_run("ls -la /proc/$(pgrep -f serial_bridge.py)/fd 2>/dev/null | grep tty || echo 'not running or no tty fd'")
print(out or err)
print()
print("If you see a /dev/ttyUSB* or /dev/serial/ path: serial_bridge.py holds the port")
print("permanently. You need the SIGSTOP workaround before scanning (see MQTT_AL1342_SETUP.md).")
print()
print("If 'not running' or no tty fd: the port is opened lazily — no extra steps needed.")

---

## Step 8 — Run a topographic scan (updated: unified gantry_agent.py architecture)

`TopographicProfiler` no longer deploys a separate script. Since later
2026-07-28, `gantry_agent.py` handles scanning itself, on a background
thread, using the same serial connection as interactive commands (see
Step 7's update). `scan()` now:
1. Calls `gantry.connection.start_scan(...)` — the agent immediately acks
   with `start_pos_mm`/`accel_mm_s2`/`decel_mm_s2` and runs the scan in the
   background
2. Blocks in `wait_for_scan_result()` until the agent reports done (or errors)
3. Retrieves the CSV/metadata sidecar via a small SFTP session (still needed —
   the files live on the Pi's disk either way)

**New: call `profiler.stop()` from another thread/cell to cancel a scan in
progress** — sends `BST` on the scanning axis; the scan still finishes
normally through the same completion path, just with fewer samples.

**Before running:** move the gantry to a safe starting position manually.
The agent reads the current position as `start_pos_mm` and moves to
`end_mm` at `feed_rate_mm_s`. Requires `transport: pi_agent` in the gantry
config (see Step 7).

In [ ]:
import sys
sys.path.insert(0, "/home/eric/mysoftware/laguna/src")

from laguna.config import Config
from laguna.robot.macron import GantryController
from laguna.robot.macron.profiler import TopographicProfiler

config = Config("config/example_config.yaml")
# Requires transport: pi_agent in the gantry: config section (not socket_bridge) —
# see Step 7. connect() SFTPs + launches gantry_agent.py itself, no manual Pi-side step.
gantry = GantryController.from_config(config.get("gantry"))
gantry.connection.connect()

profiler = TopographicProfiler(
    gantry=gantry,
    pi_host="red.lab",
    pi_user="oak",
    pi_key="/home/eric/.ssh/id_ed25519",   # still used for the SFTP CSV-retrieval session
    pdin_port=PDIN_PORT,
    al1342_host="192.168.1.251",           # raw IP — the AL1342 has no DNS of its own
    output_dir="/tmp/laguna_profiles",
)

print("Profiler ready. Edit and run the next cell to start a scan.")
print("Call profiler.stop() from another cell/thread to cancel a scan in progress.")

In [ ]:
# Adjust these before running
SCAN_AXIS        = "A1"    # X axis = A1, Y axis = A2
SCAN_END_MM      = 300.0   # target position in mm (absolute)
SCAN_RATE_MM_S   = 5.0     # slew speed — determines spatial resolution

result = profiler.scan(axis=SCAN_AXIS, end_mm=SCAN_END_MM, feed_rate_mm_s=SCAN_RATE_MM_S)

print(f"Samples collected : {result.metadata['samples']}")
print(f"Achieved rate     : {result.metadata.get('achieved_rate_hz', '?'):.1f} Hz")
print(f"Start position    : {result.metadata['actual_start_mm']:.2f} mm")
print(f"End position      : {result.metadata['actual_end_mm']:.2f} mm")
print(f"Actual distance   : {result.metadata['actual_distance_mm']:.2f} mm")
print(f"CSV path          : {result.path}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = result.df
slew = df[df["in_ramp"] == 0]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(slew["pos_mm"], slew["distance_mm"], linewidth=0.8, label="slew phase")
ax.plot(df[df["in_ramp"]==1]["pos_mm"], df[df["in_ramp"]==1]["distance_mm"],
        ".", alpha=0.3, markersize=3, label="ramp phase (excluded)")
ax.set_xlabel("Gantry position (mm)")
ax.set_ylabel("OD2000 distance (mm)")
ax.set_title(f"Topographic profile — {SCAN_AXIS} axis")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nSlew-phase samples : {len(slew)}")
print(f"Spatial resolution : {SCAN_RATE_MM_S / result.metadata.get('achieved_rate_hz', 10):.2f} mm/sample")

---

## Open items and decision points

### Resolved 2026-07-27

- **AL1342 control mechanism**: HTTP POST to `http://192.168.1.251/`, not an MQTT command channel. Confirmed working.
- **`brokerPort` type**: must be an integer (`{"newvalue": 1883}`), not a string.
- **Bootstrap path**: `/connections/mqttConnection/mqttCmdChannel/...` — no `MQTTSetup` segment.
- **AL1342 needs raw IPs**: no DNS resolution on the device itself; use `192.168.1.58`, never `red.lab`, in any config sent to the AL1342.
- **Timer floor**: confirmed by measurement at 500 ms / 2.00 Hz (24 msgs / 12.0 s) using `processdatamaster/temperature`.
- **Event-driven paths (`portevent`, `iolinkevent`) have no rate floor**, but only fire on discrete state transitions — not usable for continuous process data.

### Resolved 2026-07-28

- **OD2000 port**: confirmed on IO-Link port 2 (`productname=OD2000-7002T15`, `vendorid=26` — not 85 as guessed).
- **`pdin` datachanged**: confirmed absent, same pattern as `temperature`. MQTT push for OD2000 capped at 2 Hz by design.
- **PDIN byte layout**: confirmed big-endian int32 nm, validated against a physical 808.4mm ± 0.1mm reference (decoded 808.2778mm). Byte 4 ("scale") = 247, not 0 as assumed — unexplained, unused, doesn't affect correctness.
- **OD2000 data collection strategy**: switched from MQTT subscribe to HTTP polling of `pdin/getdata` over a persistent connection. Measured 380.7 Hz, zero errors — ~190x faster than the MQTT timer path. `scan_runner.py` and `TopographicProfiler` rewritten accordingly (`od2000_topic` → `al1342_host`).
- **Modbus TCP and hardware capture-latch**: considered as alternatives, not needed — HTTP polling alone comfortably exceeds requirements.

### Still open

**serial_bridge.py port holding**
Document result of Step 7 here once checked — next step.

In [ ]:
# Fill in findings as you go
findings = {
    "serial_bridge_holds_port": None,    # True/False — from Step 7, still open
    # Confirmed 2026-07-27:
    "al1342_control_mechanism": "HTTP POST",
    "al1342_needs_raw_ip": True,
    "al1342_timer_floor_ms": 500,        # confirmed by measurement (2.00 Hz)
    "al1342_static_ip": "192.168.1.251",
    "pi_ip": "192.168.1.58",
    # Confirmed 2026-07-28:
    "od2000_port": 2,
    "od2000_vendorid": 26,               # not 85 as originally guessed
    "pdin_byte_order": "big-endian-nm",
    "pdin_scale_byte_observed": 247,     # unexplained, unused, doesn't affect correctness
    "pdin_has_own_datachanged": False,
    "od2000_mqtt_timer_rate_hz": 2.0,    # ceiling via timer[1] subscribe
    "od2000_http_polling_rate_hz": 380.7,  # confirmed via persistent connection, zero errors
    "scan_data_collection_method": "http_polling",  # not mqtt
}
print(findings)

---

## NFS vs SFTP note

`TopographicProfiler` still uses SFTP for the CSV/metadata retrieval step
(the scan itself no longer needs SFTP — no script is deployed for it, only
`gantry_agent.py` on connect, unchanged from before). If you set up an NFS
mount of the Pi's output directory on the laguna PC, retrieval can be
skipped — the CSV appears as a local file automatically.

To use NFS: add `nfs_output_path="/mnt/pi/profiles"` to the `TopographicProfiler`
constructor, and update `profiler.py`'s `_sftp_retrieve` to read from that
path instead. Not implemented yet — add it once the NFS mount is set up.

---

## What to merge when done

Branch `feat-mqtt-od2000-integration` → `develop`.

Before merging:
- [ ] All 308 tests still pass
- [ ] Steps 1–7 above verified on hardware (done, except Step 8's real scan)
- [ ] Step 8: run one real scan on hardware — confirm `start_scan`/`wait_for_scan_result`
      work against the real agent, confirm `stop_scan()` cancels within ~one MIF
      poll tick (~0.1s), confirm CSV/metadata retrieval
- [ ] `findings` dict above filled in
- [ ] At least one real profile CSV in the repo or linked from the docs as a reference

After merging, open items for follow-up branches:
- Approach B: wire OD2000 Q2/Qa → INB 7 for hardware-triggered capture-latch (higher positional fidelity) — not needed given 380 Hz HTTP polling, but still documented as a fallback
- MQTT auth: add Mosquitto credentials if the lab network ever changes
- NFS output directory if preferred over SFTP retrieval
- Gauge publisher wiring: deploy `gauge_publisher.py` as a persistent service rather than on-demand